# OpenVSP: size the plane with OpenVSP parasite drag

`run_pipeline` sizes the plane with `evaluate`, gives the wing (area, aspect ratio, taper) to the OpenVSP model, runs the parasite drag build-up and feeds that $C_{D,0}$ back into `evaluate`, until $C_{D,0}$ stops changing.

The `.vsp3` file is set in `ardea_sizer/pipeline/_open_vsp.py` and stays unchanged on disk unless `SAVE_AS` is set. $C_{D,0}$ is referenced to the wing area, like `c_d_0` in `base.json`.

In [11]:
from pathlib import Path

import pandas as pd
import openvsp as vsp

import ardea_sizer as az
from ardea_sizer.pipeline._open_vsp import VSP3_FILE, WING_NAME
from ardea_sizer.pipeline._pipeline import run_pipeline

BASE_FILE = "base.json"
SAVE_AS = None                # e.g. "models/20261012_Convent_BM_V3_sized.vsp3"

In [12]:
def wing_summary() -> pd.Series:
    wing_id = vsp.FindGeomsWithName(WING_NAME)[0]
    g = lambda name, group: vsp.GetParmVal(wing_id, name, group)
    return pd.Series({
        "area [m²]": g("TotalArea", "WingGeom"),
        "aspect ratio": g("TotalAR", "WingGeom"),
        "span [m]": g("TotalSpan", "WingGeom"),
        "mean chord [m]": g("TotalChord", "WingGeom"),
        "MAC [m]": g("MAC", "WingGeom"),
        "root chord [m]": g("Root_Chord", "XSec_1"),
        "tip chord [m]": g("Tip_Chord", "XSec_1"),
        "taper": g("Taper", "XSec_1"),
    })


def drag_components() -> pd.DataFrame:
    # last parasite drag run (the one inside optimize)
    rid = vsp.FindLatestResultsID("Parasite_Drag")
    return pd.DataFrame({
        "S_wet [m²]": vsp.GetDoubleResults(rid, "Comp_Swet", 0),
        "f [m²]": vsp.GetDoubleResults(rid, "Comp_f", 0),
        "CD": vsp.GetDoubleResults(rid, "Comp_CD", 0),
    }, index=vsp.GetStringResults(rid, "Comp_Label", 0))

## Wing in the file (before sizing)

In [13]:
vsp.ClearVSPModel()
vsp.ReadVSPFile(str(VSP3_FILE))
vsp.Update()
wing_file = wing_summary()
wing_file.round(4).to_frame(VSP3_FILE.name)

,20261012_Convent_BM_V3.vsp3
area [m²],0.4000
aspect ratio,8.0000
span [m],1.7889
mean chord [m],0.2236
MAC [m],0.2343
root chord [m],0.3084
tip chord [m],0.1388
taper,0.4500


## Sizing loop with $C_{D,0}$ feedback

In [14]:
base_start = az.load(BASE_FILE)
plane, base = run_pipeline(base_start)

pd.Series({
    "c_d_0 start": base_start.assumptions.c_d_0,
    "c_d_0 converged": base.assumptions.c_d_0,
    "fuel mass [kg]": float(plane.fuel_mass),
    "wing area [m²]": float(plane.wing_area),
    "aspect ratio": float(plane.aspect_ratio),
    "taper": float(plane.taper),
    "range [km]": float(plane.range) / 1000,
    "power required [kW]": float(plane.power_req) / 1000,
    "power available [kW]": float(plane.power_available),
}).round(4).to_frame("sized plane")

Run 1: c_d_0 0.02200 -> 0.01565   S 0.5062 m^2   fuel 8.090 kg
Run 2: c_d_0 0.01565 -> 0.01677   S 0.4366 m^2   fuel 5.051 kg
Run 3: c_d_0 0.01677 -> 0.01657   S 0.4475 m^2   fuel 5.527 kg
Run 4: c_d_0 0.01657 -> 0.01661   S 0.4455 m^2   fuel 5.441 kg


,sized plane
c_d_0 start,0.0220
c_d_0 converged,0.0166
fuel mass [kg],5.4408
wing area [m²],0.4455
aspect ratio,5.0000
taper,0.4500
range [km],1050.0131
power required [kW],3.4844
power available [kW],4.9700


## Sized wing in OpenVSP

The model in memory now holds the wing of the last run. Parasite drag per component ($C_D$ referenced to the sized wing area):

In [15]:
display(drag_components().round(5))

wing_sized = wing_summary()
comparison = pd.DataFrame({"file": wing_file, "sized": wing_sized})
comparison["change [%]"] = 100 * (comparison["sized"] / comparison["file"] - 1)
comparison.round(4)

,S_wet [m²],f [m²],CD
ARDEA_D3_Stack,0.76725,0.00268,0.00602
vertical stabilizer,0.06139,0.00039,0.00087
Horizontal stabilizer,0.11930,0.00075,0.00169
Main Wing,0.71561,0.00358,0.00803


,file,sized,change [%]
area [m²],0.4000,0.4455,11.3800
aspect ratio,8.0000,5.0000,-37.5000
span [m],1.7889,1.4925,-16.5659
mean chord [m],0.2236,0.2985,33.4946
MAC [m],0.2343,0.3128,33.4946
root chord [m],0.3084,0.4117,33.4946
tip chord [m],0.1388,0.1853,33.4946
taper,0.4500,0.4500,0.0000


In [16]:
if SAVE_AS:
    vsp.WriteVSPFile(str(Path(SAVE_AS).resolve()))
    print(f"Saved to {SAVE_AS}")